### **LightGBM Regression – Dự đoán thành tính học tập của sinh viên**
Đọc dữ liệu → làm sạch → lưu dữ liệu sạch → chia Train/Test 80/20 → huấn luyện LightGBM → đánh giá → trực quan hóa. 




In [1]:

import sys
from pathlib import Path

# Thêm đường dẫn cần thiết vào sys.path để import được LightGBMRegression
# (hỗ trợ chạy notebook từ bất kỳ đâu: regression/, notebooks/, repo root...).
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)
REPO_ROOT = next(
    (
        path for path in repo_candidates
        if (path / "regression" / "lightgbm_regression.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy repo root chứa regression/lightgbm_regression.py."
    )

REGRESSION_DIR = REPO_ROOT / "regression"

# Thêm cả repo root (cho import theo package regression.*)
# và thư mục regression (cho import trực tiếp lightgbm_regression)
# vào sys.path để import hoạt động từ bất kỳ thư mục nào.
for path in (REPO_ROOT, REGRESSION_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer  # Hỗ trợ xử lý dữ liệu bị thiếu
from sklearn.preprocessing import OrdinalEncoder  # Mã hóa categorical nhiều cột
from lightgbm_regression import LightGBMRegression

from regression.metrics import (
    evaluate_regression,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from regression.visualization import (
    plot_actual_vs_predicted,
    plot_residuals,
    plot_error_distribution,
    plot_train_test_metrics_comparison,
    plot_feature_importance,
)
from regression.evaluation import evaluate_and_visualize

print("Import LightGBMRegression thành công.")
print("REPO_ROOT =", REPO_ROOT)
print("REGRESSION_DIR =", REGRESSION_DIR)


Import LightGBMRegression thành công.
REPO_ROOT = D:\machine-learning-group-6
REGRESSION_DIR = D:\machine-learning-group-6\regression


In [2]:
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)

REPO_ROOT = next(
    (
        path for path in repo_candidates
        if (path / "regression").is_dir()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy repo root chứa thư mục regression."
    )

repo_root_text = str(REPO_ROOT)

if repo_root_text not in sys.path:
    sys.path.insert(0, repo_root_text)

print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = D:\machine-learning-group-6


####  **Data processing**

In [3]:
DATA_PATH = (
    REPO_ROOT
    / "regression" / "data" / "raw" / "student_exam_performance.csv"
)

# Đọc dữ liệu  
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

df.head(10)

Shape: (100000, 44)


,student_id,age,gender,education_level,school_type,family_income,parent_education,urban_rural,previous_exam_score,previous_gpa,...,exam_difficulty,exam_preparation_days,questions_attempted,questions_correct,time_management_score,exam_anxiety_level,exam_score,performance_grade,pass_status,performance_level
0,STU_000001,20,Female,High School,Public,Middle,NaN,Rural,78.36,2.95,...,Medium,27,95,86,NaN,9.39,90.40,A,Pass,High
1,STU_000002,17,Male,High School,Public,Middle,NaN,Suburban,73.40,2.89,...,Easy,4,100,86,87.17,6.31,86.21,A,Pass,High
2,STU_000003,18,Other,Undergraduate,Public,Middle,Master,Urban,82.76,3.35,...,Easy,28,96,74,63.77,9.87,76.11,B,Pass,Medium
3,STU_000004,20,Male,High School,Public,Middle,High School,Suburban,60.61,2.57,...,Medium,16,97,77,59.51,9.68,79.74,B,Pass,Medium
4,STU_000005,16,Female,High School,Private,High,Bachelor,Suburban,74.79,2.84,...,Medium,28,98,52,84.64,10.00,50.66,D,Pass,Low
5,STU_000006,18,Male,High School,Public,High,High School,Rural,67.68,NaN,...,Hard,27,96,65,55.27,6.12,67.90,C,Pass,Medium
6,STU_000007,18,Male,High School,Charter,Middle,NaN,Urban,78.84,3.00,...,Hard,12,98,54,86.15,9.46,53.86,D,Pass,Low
7,STU_000008,20,Female,Undergraduate,Private,Lower-Middle,Bachelor,Suburban,90.79,3.63,...,Easy,7,94,91,88.08,10.00,97.61,A,Pass,High
8,STU_000009,15,Male,Undergraduate,Public,Lower-Middle,Associate,Suburban,61.28,2.43,...,Hard,14,94,35,70.21,8.34,34.44,F,Fail,Low
9,STU_000010,16,Female,Undergraduate,Public,Lower-Middle,Associate,Rural,61.33,2.72,...,Hard,4,87,34,73.78,8.49,39.05,F,Fail,Low


In [4]:
# Kiểm tra thông tin data 
df.info ( )

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 44 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   student_id                  100000 non-null  object 
 1   age                         100000 non-null  int64  
 2   gender                      100000 non-null  object 
 3   education_level             100000 non-null  object 
 4   school_type                 100000 non-null  object 
 5   family_income               100000 non-null  object 
 6   parent_education            93474 non-null   object 
 7   urban_rural                 100000 non-null  object 
 8   previous_exam_score         100000 non-null  float64
 9   previous_gpa                92207 non-null   float64
 10  attendance_percentage       90137 non-null   float64
 11  assignment_completion_rate  100000 non-null  float64
 12  class_participation         100000 non-null  object 
 13  study_hours_per

In [5]:
# Kiểm tra dữ liệu bị null 
df.isnull().sum()

student_id                       0
age                              0
gender                           0
education_level                  0
school_type                      0
family_income                    0
parent_education              6526
urban_rural                      0
previous_exam_score              0
previous_gpa                  7793
attendance_percentage         9863
assignment_completion_rate       0
class_participation              0
study_hours_per_day              0
self_study_hours                 0
private_tuition                  0
online_learning_hours            0
study_consistency                0
study_environment                0
study_method                     0
revision_frequency               0
practice_tests_completed         0
notes_quality                 8407
sleep_hours                      0
sleep_quality                 6994
daily_screen_time                0
physical_activity_hours          0
break_frequency                  0
stress_level        

#### **Loại bỏ các cột không phù hợp**
#### **Loại student_id vì đây chỉ là mã định danh.**
#### **Loại performance_grade, pass_status và performance_level vì các cột này được suy ra từ exam_score, dễ gây data**  **leakage trong bài toán Regression.**

In [6]:
drop_columns = [
    "student_id",
    "performance_grade",
    "pass_status",
    "performance_level"
]

df = df.drop(columns=drop_columns)

# Hình dạng dataset sau xóa  
print("Shape:", df.shape)

Shape: (100000, 40)


In [7]:
#  Kiểm tra số lượng dòng theo từng feature 
df.isnull().sum()[df.isnull().sum() > 0]

parent_education         6526
previous_gpa             7793
attendance_percentage    9863
notes_quality            8407
sleep_quality            6994
device_availability      2965
time_management_score    9613
dtype: int64

In [8]:
# Kiểm tra kiểu dữ liệu của các feature đang bị thiếu
missing_columns = df.columns[df.isnull().any()].tolist()

df[missing_columns].dtypes

parent_education          object
previous_gpa             float64
attendance_percentage    float64
notes_quality             object
sleep_quality             object
device_availability       object
time_management_score    float64
dtype: object

In [9]:
# Tự động phân loại cột dựa vào kiểu dữ liệu (dtype)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Điền cột số bằng Mean (giá trị trung bình)
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# Điền cột categorical bằng Mode (giá trị xuất hiện nhiều nhất)
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
# Kiểm tra missing sau khi xử lý
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [11]:
# Xác định các feature categorical
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(cat_cols)

['gender', 'education_level', 'school_type', 'family_income', 'parent_education', 'urban_rural', 'class_participation', 'study_consistency', 'study_environment', 'study_method', 'revision_frequency', 'notes_quality', 'sleep_quality', 'break_frequency', 'motivation_level', 'device_availability', 'educational_app_usage', 'exam_difficulty']


In [12]:
# Chuyển categorical thành dữ liệu số
encoder = OrdinalEncoder()

df[cat_cols] = encoder.fit_transform(
    df[cat_cols]
)

In [13]:
# Kiểm tra dữ liệu sau khi encode
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   age                         100000 non-null  int64  
 1   gender                      100000 non-null  float64
 2   education_level             100000 non-null  float64
 3   school_type                 100000 non-null  float64
 4   family_income               100000 non-null  float64
 5   parent_education            100000 non-null  float64
 6   urban_rural                 100000 non-null  float64
 7   previous_exam_score         100000 non-null  float64
 8   previous_gpa                100000 non-null  float64
 9   attendance_percentage       100000 non-null  float64
 10  assignment_completion_rate  100000 non-null  float64
 11  class_participation         100000 non-null  float64
 12  study_hours_per_day         100000 non-null  float64
 13  self_study_hour

In [14]:
# Lưu dữ liệu sau khi xử lý 
df.to_csv(r"..\data\processed\student_performance_processed.csv", index=False)

### **Traning and evalute model**

In [15]:
# Đọc dữ liệu đã tiền xử lý
df_processed = pd.read_csv(r"../data/processed/student_performance_processed.csv")

print(df_processed.shape)
df_processed.head()

(100000, 40)


,age,gender,education_level,school_type,family_income,parent_education,urban_rural,previous_exam_score,previous_gpa,attendance_percentage,...,device_availability,educational_app_usage,online_course_hours,exam_difficulty,exam_preparation_days,questions_attempted,questions_correct,time_management_score,exam_anxiety_level,exam_score
0,20,0.0,0.0,2.0,3.0,3.0,0.0,78.36,2.95,81.490000,...,1.0,1.0,2.15,2.0,27,95,86,78.398551,9.39,90.40
1,17,1.0,0.0,2.0,3.0,3.0,1.0,73.40,2.89,84.616882,...,1.0,2.0,4.37,0.0,4,100,86,87.170000,6.31,86.21
2,18,2.0,1.0,2.0,3.0,4.0,2.0,82.76,3.35,100.000000,...,0.0,1.0,1.43,0.0,28,96,74,63.770000,9.87,76.11
3,20,1.0,0.0,2.0,3.0,3.0,1.0,60.61,2.57,84.910000,...,0.0,1.0,3.69,2.0,16,97,77,59.510000,9.68,79.74
4,16,0.0,0.0,1.0,0.0,1.0,1.0,74.79,2.84,88.210000,...,0.0,2.0,1.97,2.0,28,98,52,84.640000,10.00,50.66


In [16]:
# Feature và target 
X = df_processed.drop(columns=["exam_score"])
y = df_processed["exam_score"]

print("X:", X.shape)
print("y:", y.shape)

X: (100000, 39)
y: (100000,)


In [17]:
# Chia dữ liệu thành train và test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (80000, 39)
X_test: (20000, 39)
y_train: (80000,)
y_test: (20000,)


In [18]:
# Khởi tạo mô hình
model = LightGBMRegression(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

In [19]:
# Train model  
model.fit ( X_train , y_train ) 

Đang tiến hành huấn luyện (fit)...
Huấn luyện hoàn tất!


In [20]:
# Dự đoán trên cả tập train và test
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
print("y_train_pred:", y_train_pred.shape)
print("y_test_pred:", y_test_pred.shape)

Đang tiến hành dự đoán (predict)...
Đang tiến hành dự đoán (predict)...
y_train_pred: (80000,)
y_test_pred: (20000,)


In [21]:
# Đánh giá tự động + xuất toàn bộ biểu đồ (metrics, CSV, 8 ảnh)
feature_names = (
    list(X_train.columns)
    if hasattr(X_train, "columns")
    else [f"feature_{i}" for i in range(X_train.shape[1])]
)

results = evaluate_and_visualize(
    y_train, y_train_pred,
    y_test, y_test_pred,
    model=model,
    feature_names=feature_names,
)
results

ĐÁNH GIÁ REGRESSION - TRAIN & TEST
Split            MAE           MSE          R2
--------------------------------------------------------
train       1.441448      3.411977    0.986483
test        1.510870      4.005270    0.983888
Đã lưu kết quả vào: D:\machine-learning-group-6\regression\outputs\result\train_test_result.csv
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\train_actual_vs_predicted.png
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\test_actual_vs_predicted.png
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\train_residual_plot.png
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\test_residual_plot.png


c:\Users\LENOVO\anaconda3\envs\ai_machine_learning\lib\site-packages\seaborn\_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):


Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\train_error_distribution.png


c:\Users\LENOVO\anaconda3\envs\ai_machine_learning\lib\site-packages\seaborn\_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):


Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\test_error_distribution.png
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\train_test_metrics_bar_chart.png
Đã lưu ảnh: D:\machine-learning-group-6\regression\outputs\figures\feature_importance.png
Đã xuất toàn bộ ảnh vào: D:\machine-learning-group-6\regression\outputs\figures


{'train': {'MAE': 1.441448283042448,
  'MSE': 3.4119774251461337,
  'R2': 0.986482536753169},
 'test': {'MAE': 1.510870038516188,
  'MSE': 4.005269867459568,
  'R2': 0.9838878215334811}}